In [0]:
import pandas as pd


In [0]:
def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    dt_nasc = pd.to_datetime(df['DT_NASC'], errors='coerce')
    dt_sint = pd.to_datetime(df['DT_SIN_PRI'], errors='coerce')

    df['idade'] = ((dt_sint - dt_nasc).dt.days / 365.25).round(1)

    map_evol = {1: 'Cura', 2: 'Óbito', 3: 'Óbito outras causas', 9: 'Ignorado',
                '1': 'Cura', '2': 'Óbito', '3': 'Óbito outras causas', '9': 'Ignorado'}
    map_uti = {1: 'Sim', 2: 'Não', 9: 'Ignorado',
               '1': 'Sim', '2': 'Não', '9': 'Ignorado'}
    map_vac = {1: 'Sim', 2: 'Não', 9: 'Ignorado',
               '1': 'Sim', '2': 'Não', '9': 'Ignorado'}

    df['evolucao'] = df['EVOLUCAO'].map(map_evol)
    df['uti'] = df['UTI'].map(map_uti)
    df['vacina_covid'] = df['VACINA_COV'].map(map_vac)

    rename_map = {
        'DT_SIN_PRI': 'data_sintomas',
        'SG_UF': 'uf',
        'DT_ENTUTI': 'data_entrada_uti',
        'DT_SAIDUTI': 'data_saida_uti',
        'DOSE_1_COV': 'data_dose1_covid'
    }
    df = df.rename(columns=rename_map)

    cols = ['data_sintomas', 'uf', 'idade', 'evolucao', 'uti', 'vacina_covid',
            'data_entrada_uti', 'data_saida_uti', 'data_dose1_covid']
    df = df[cols]

    df = df.dropna(subset=['data_sintomas', 'uf', 'idade', 'evolucao', 'uti'])
    df['idade'] = df['idade'].fillna(-1).astype(int)
    df = df.query('0 <= idade <= 120')
    df['uf'] = df['uf'].astype(str).str.upper().str.strip()

    for c in ['data_sintomas', 'data_entrada_uti', 'data_saida_uti', 'data_dose1_covid']:
        df[c] = pd.to_datetime(df[c], errors='coerce')

    return df

In [0]:
df_raw = spark.table("srag_datasus.raw.srag_data").toPandas()

df_clean = prepare_dataframe(df_raw)

df_silver = spark.createDataFrame(df_clean)

df_silver.write.format("delta").mode("overwrite").saveAsTable("srag_datasus.silver.int_srag")